# Comparação GraphRAG vs RAG ingênuo — dataset `rag_eval_qa_estruturado.csv`

Mesmas métricas do notebook `comparacao_graphrag_vs_naive_rag.ipynb`:

**Recuperação (top-5)**
- **Recall@5** e **Precision@5** em relação aos `TextUnit` cujo texto contém os trechos de evidência (`trecho_1` / `trecho_2`) nos ficheiros de `fonte_trecho_*`, usando o **mesmo chunking** que `graphrag.indexing.load_and_chunk.chunk_documents`.

**Geração**
- **F1 por tokens** entre resposta gerada e `resposta_esperada`.

**CSV** — `docs/rag_eval_qa_estruturado.csv` (colunas `pergunta`, `tipo_pergunta`, `resposta_esperada`, trechos e fontes).

**Pré-requisitos** — Neo4j a correr, `.env` com `OPENAI_API_KEY`. A **secção 5** indexa o KG a partir de `pdfs_txt/` (obrigatório antes do GraphRAG / índice vetorial); depois execute a **secção 6** (comparação).

**Se `OPENAI_API_KEY` aparecer vazia:** a primeira célula lê o ficheiro `graphrag-neo4j-langchain/.env` mesmo **sem** `python-dotenv` (fallback manual). Ainda assim recomenda-se `pip install python-dotenv` no kernel. Reinicie o kernel e execute do topo se já tiver importado `graphrag` noutra ordem. Linha no `.env`: `OPENAI_API_KEY=sk-...` (sem aspas; confirme que o prefixo é `sk-` e não `ssk-`).


In [1]:
from __future__ import annotations

import csv
import re
import sys
from collections import Counter
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path

_here = Path.cwd()
if (_here / "graphrag-neo4j-langchain" / "src").is_dir():
    PROJECT_ROOT = _here.resolve()
elif (_here.parent / "graphrag-neo4j-langchain" / "src").is_dir():
    PROJECT_ROOT = _here.parent.resolve()
else:
    raise FileNotFoundError(
        "Defina manualmente PROJECT_ROOT para a pasta que contém graphrag-neo4j-langchain/ "
        f"(cwd atual: {_here})"
    )

GRAPHRAG_SRC = PROJECT_ROOT / "graphrag-neo4j-langchain" / "src"
if not GRAPHRAG_SRC.is_dir():
    raise FileNotFoundError(f"Não encontrei graphrag em: {GRAPHRAG_SRC}")

if str(GRAPHRAG_SRC) not in sys.path:
    sys.path.insert(0, str(GRAPHRAG_SRC))

# Caminho do .env (mesmo usado pelo pacote em graphrag/config.py)
_ENV_FILE = PROJECT_ROOT / "graphrag-neo4j-langchain" / ".env"


def _parse_env_file(path: Path) -> dict[str, str]:
    """Lê KEY=VAL do .env sem depender de python-dotenv (fallback para o kernel Jupyter)."""
    if not path.is_file():
        return {}
    try:
        raw = path.read_text(encoding="utf-8-sig")
    except OSError:
        return {}
    out: dict[str, str] = {}
    for line in raw.splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        if "=" not in s:
            continue
        key, _, rest = s.partition("=")
        key = key.strip()
        val = rest.strip().strip('"').strip("'")
        if key:
            out[key] = val
    return out


def load_graphrag_env() -> None:
    """Carrega variáveis no os.environ e refresca graphrag.config se já importado."""
    import importlib
    import os

    try:
        from dotenv import load_dotenv

        load_dotenv(_ENV_FILE, override=True)
    except ImportError:
        pass

    # Fallback: mesmo sem python-dotenv, injeta o .env (resolve kernel sem o pacote)
    for k, v in _parse_env_file(_ENV_FILE).items():
        if v:
            os.environ[k] = v

    if "graphrag.config" in sys.modules:
        import graphrag.config

        importlib.reload(graphrag.config)


load_graphrag_env()
if not _ENV_FILE.is_file():
    print(f"Aviso: ficheiro .env não encontrado em {_ENV_FILE} (pode usar OPENAI_API_KEY no ambiente do sistema).")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OPENAI_API_KEY no ambiente:", bool(__import__("os").environ.get("OPENAI_API_KEY")))

PROJECT_ROOT: /home/micael/mestrado/benchmarking-graphrag
OPENAI_API_KEY no ambiente: True


## 1. Métricas (iguais ao notebook de referência)


In [2]:
def normalize_tokens(text: str) -> list[str]:
    t = (text or "").lower()
    t = re.sub(r"[^\w\s\u00C0-\u024F]", " ", t, flags=re.UNICODE)
    return [x for x in t.split() if x]


def token_f1(reference: str, predicted: str) -> float:
    """F1 por tokens (multiset)."""
    ref = normalize_tokens(reference)
    hyp = normalize_tokens(predicted)
    if not ref and not hyp:
        return 1.0
    if not ref or not hyp:
        return 0.0
    cr, ch = Counter(ref), Counter(hyp)
    overlap = sum((cr & ch).values())
    prec = overlap / max(len(hyp), 1)
    rec = overlap / max(len(ref), 1)
    if prec + rec == 0:
        return 0.0
    return 2 * prec * rec / (prec + rec)


def recall_at_k(retrieved_ids: list[str], gold_ids: set[str], k: int) -> float:
    top = retrieved_ids[:k]
    hits = sum(1 for x in top if x in gold_ids)
    if not gold_ids:
        return 1.0
    return hits / len(gold_ids)


def precision_at_k(retrieved_ids: list[str], gold_ids: set[str], k: int) -> float:
    top = retrieved_ids[:k]
    if k <= 0:
        return 0.0
    hits = sum(1 for x in top if x in gold_ids)
    return hits / k

## 2. CSV estruturado → `TextUnit.id` do gabarito (trechos × chunking)


In [3]:
@dataclass
class QARowEstruturado:
    qid: str
    tipo_pergunta: str
    question: str
    trecho_1: str
    fonte_trecho_1: str
    trecho_2: str
    fonte_trecho_2: str
    reference_answer: str


def load_qa_estruturado_csv(path: Path) -> list[QARowEstruturado]:
    rows: list[QARowEstruturado] = []
    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        need = {
            "id",
            "tipo_pergunta",
            "pergunta",
            "trecho_1",
            "fonte_trecho_1",
            "trecho_2",
            "fonte_trecho_2",
            "resposta_esperada",
        }
        miss = need - set(reader.fieldnames or [])
        if miss:
            raise ValueError(f"CSV incompleto. Faltam colunas: {sorted(miss)}")
        for r in reader:
            q = (r.get("pergunta") or "").strip()
            ra = (r.get("resposta_esperada") or "").strip()
            qid = (r.get("id") or "").strip()
            if not q or not ra:
                continue
            rows.append(
                QARowEstruturado(
                    qid=qid,
                    tipo_pergunta=(r.get("tipo_pergunta") or "").strip(),
                    question=q,
                    trecho_1=(r.get("trecho_1") or "").strip(),
                    fonte_trecho_1=(r.get("fonte_trecho_1") or "").strip(),
                    trecho_2=(r.get("trecho_2") or "").strip(),
                    fonte_trecho_2=(r.get("fonte_trecho_2") or "").strip(),
                    reference_answer=ra,
                )
            )
    if not rows:
        raise ValueError("Nenhuma linha válida no CSV.")
    return rows


def _norm_for_match(s: str) -> str:
    s = (s or "").lower()
    s = re.sub(r"<br\s*/?>", " ", s, flags=re.IGNORECASE)
    s = re.sub(r"\s+", " ", s)
    return s.strip()


@lru_cache(maxsize=64)
def _chunk_texts_and_tu_ids(resolved_path: str) -> tuple[tuple[str, ...], tuple[str, ...]]:
    from langchain_core.documents import Document
    from graphrag.indexing.load_and_chunk import chunk_documents

    doc_path = Path(resolved_path)
    text = doc_path.read_text(encoding="utf-8", errors="ignore")
    doc = Document(page_content=text, metadata={"source": str(doc_path)})
    chunks = chunk_documents([doc])
    base = str(doc_path.resolve())
    texts = tuple(c.page_content for c in chunks)
    ids = tuple(f"{base}_{i}" for i in range(len(chunks)))
    return texts, ids


def _find_tu_ids_for_snippet(doc_path: Path, snippet: str) -> set[str]:
    # TextUnits cujo texto contém o trecho (heurística por substring / tokens).
    path = str(doc_path.resolve())
    texts, tu_ids = _chunk_texts_and_tu_ids(path)
    if not snippet.strip():
        return set()

    ns = _norm_for_match(snippet).replace("...", " ")
    ns = re.sub(r"\s+", " ", ns).strip()
    out: set[str] = set()

    for i, raw in enumerate(texts):
        nc = _norm_for_match(raw)
        if len(ns) >= 40:
            for win in (ns[:120], ns[:80], ns[-80:] if len(ns) > 80 else ""):
                if win and len(win) >= 25 and win in nc:
                    out.add(tu_ids[i])
                    break
            else:
                pass
        elif ns and len(ns) < 40:
            if ns in nc or (len(ns) >= 12 and ns[:25] in nc):
                out.add(tu_ids[i])

    if out:
        return out

    # fallback: tokens/números significativos
    tokens = re.findall(r"[a-z0-9]{3,}|[0-9]{4,}", ns)
    for i, raw in enumerate(texts):
        nc = _norm_for_match(raw)
        if not tokens:
            continue
        hit = sum(1 for t in tokens if t in nc)
        if hit >= max(1, min(3, len(tokens) // 2)):
            out.add(tu_ids[i])
    return out


def gold_text_unit_ids_for_row(row: QARowEstruturado, docs_dir: Path) -> set[str]:
    # Reúne TU a partir de (trecho_k, fonte_k).
    gold: set[str] = set()
    for trecho, fname in ((row.trecho_1, row.fonte_trecho_1), (row.trecho_2, row.fonte_trecho_2)):
        if not fname or not str(trecho).strip():
            continue
        fp = (docs_dir / fname).resolve()
        if not fp.is_file():
            raise FileNotFoundError(f"Ficheiro de fonte inexistente: {fp}")
        gold |= _find_tu_ids_for_snippet(fp, str(trecho))
    return gold


## 3. Recuperação e geração: RAG ingênuo vs GraphRAG


In [4]:
def naive_retrieve_top_k(question: str, k: int) -> tuple[list[str], list[str]]:
    from graphrag.store.vector_index import get_vector_index_text_units

    store = get_vector_index_text_units()
    if store is None:
        raise RuntimeError("Índice vetorial TextUnit indisponível. Execute embed no pipeline.")
    docs = store.similarity_search(question.strip(), k=k)
    ids, texts = [], []
    for d in docs:
        md = getattr(d, "metadata", None) or {}
        tid = md.get("id") or md.get("t.id")
        ids.append(str(tid) if tid is not None else "")
        texts.append(getattr(d, "page_content", "") or "")
    return ids, texts


def graphrag_top_k_text_unit_ids(state: dict, k: int) -> list[str]:
    out: list[str] = []
    for doc in state.get("context_docs") or []:
        if not isinstance(doc, dict):
            continue
        md = doc.get("metadata") or {}
        if not isinstance(md, dict):
            continue
        kind = str(md.get("kind") or "")
        if not kind.startswith("text_unit"):
            continue
        tid = md.get("text_unit_id")
        if not tid:
            continue
        s = str(tid)
        if s not in out:
            out.append(s)
        if len(out) >= k:
            break
    return out[:k]


def run_graphrag(question: str, thread_id: str) -> dict:
    from graphrag.graph.query_graph import get_compiled_graph

    compiled = get_compiled_graph()
    cfg = {"configurable": {"thread_id": f"nb-estr-{thread_id}"}}
    return compiled.invoke({"question": question}, config=cfg)


def naive_generate(question: str, context_chunks: list[str]) -> str:
    from graphrag.config import OPENAI_API_KEY, LLM_MODEL
    from graphrag.monitoring.token_cost import tracked_chat_openai
    from graphrag.prompts.synthesis import SYNTHESIS_PROMPT

    if not OPENAI_API_KEY:
        raise RuntimeError("OPENAI_API_KEY não definida.")
    ctx = "\n\n".join(context_chunks) if context_chunks else "(sem contexto)"
    llm = tracked_chat_openai(model=LLM_MODEL, temperature=0, api_key=OPENAI_API_KEY)
    chain = SYNTHESIS_PROMPT | llm
    msg = chain.invoke({"context": ctx, "question": question})
    return getattr(msg, "content", str(msg)) or ""


## 4. Parâmetros e validação do ambiente


In [5]:
import os

# Garante que o .env foi aplicado antes de ler graphrag.config (e recarrega o módulo se necessário)
load_graphrag_env()

K = 5
DOCS_DIR = (PROJECT_ROOT / "pdfs_txt").resolve()
QA_CSV = (PROJECT_ROOT / "docs" / "rag_eval_qa_estruturado.csv").resolve()

assert DOCS_DIR.is_dir(), f"Falta a pasta de documentos: {DOCS_DIR}"
assert QA_CSV.is_file(), f"Falta o CSV: {QA_CSV}"

from graphrag.config import OPENAI_API_KEY

if not (OPENAI_API_KEY or "").strip():
    hint = (
        f"OPENAI_API_KEY está vazia após carregar {_ENV_FILE}. "
        "1) pip install python-dotenv  2) Confirme OPENAI_API_KEY=... no .env (sem aspas)  3) Reinicie o kernel e execute deste o topo."
    )
    raise RuntimeError(hint)

txt_files = sorted(DOCS_DIR.glob("*.txt"))
assert txt_files, f"Nenhum .txt encontrado em: {DOCS_DIR}"

qa_rows = load_qa_estruturado_csv(QA_CSV)

print(f"Pasta de entrada: {DOCS_DIR}")
print(f"CSV: {QA_CSV.name}")
print(f"Ficheiros .txt: {len(txt_files)} | Perguntas: {len(qa_rows)}")

Pasta de entrada: /home/micael/mestrado/benchmarking-graphrag/pdfs_txt
CSV: rag_eval_qa_estruturado.csv
Ficheiros .txt: 5 | Perguntas: 9


## 5. Indexação do grafo de conhecimento (KG)

Antes de avaliar o **GraphRAG** e o **RAG ingênuo**, o pipeline precisa de dados no Neo4j: `TextUnit`, entidades, relações, comunidades, relatórios e **embeddings** (índice vetorial usado na recuperação).

Execute a célula seguinte para correr o mesmo fluxo que `scripts/run_indexing.py` sobre `DOCS_DIR` (`pdfs_txt/`). Pode demorar e consumir API OpenAI.

- `RUN_KG_INDEXING = True` — executa indexação completa.
- `RUN_KG_INDEXING = False` — ignora (assume que já indexou noutra sessão).

In [8]:
load_graphrag_env()

# --- ajuste aqui ---
RUN_KG_INDEXING = True  # False para saltar se o KG já estiver indexado
CHUNK_STRATEGY_INDEX: str | None = None  # None = usa CHUNK_STRATEGY do config; ou "tokens" / "chars"

if RUN_KG_INDEXING:
    from graphrag.config import CHUNK_STRATEGY
    from graphrag.indexing.load_and_chunk import run_load_and_chunk
    from graphrag.indexing.extract_graph import run_extract_on_chunks
    from graphrag.indexing.communities import run_communities
    from graphrag.indexing.graph_links import link_claims_and_covariates_to_communities
    from graphrag.indexing.reports import run_reports
    from graphrag.indexing.embed import run_embed_all
    from graphrag.store.neo4j_graph import get_neo4j_graph
    from graphrag.monitoring.token_cost import TRACKER

    TRACKER.reset("indexing")
    strat = CHUNK_STRATEGY_INDEX or CHUNK_STRATEGY
    print(f"[KG] 1/5 Load + chunk (strategy={strat}) em {DOCS_DIR}")
    chunk_records = run_load_and_chunk(DOCS_DIR, strategy=CHUNK_STRATEGY_INDEX)
    print(f"     TextUnits criados: {len(chunk_records)}")

    if chunk_records:
        print("[KG] 2/5 Extração de entidades e relações...")
        run_extract_on_chunks(chunk_records)

    print("[KG] 3/5 Comunidades + ligações a claims/covariates...")
    comms = run_communities()
    with get_neo4j_graph()._driver.session() as session:
        total = session.run("MATCH (c:Community) RETURN count(c) AS n").single()["n"]
    print(f"     Comunidades L0: {len(comms)} | total nós Community: {total}")
    lc = link_claims_and_covariates_to_communities()
    print(f"     Links: {lc['claim_links']} claims, {lc['covariate_links']} covariates")

    print("[KG] 4/5 Relatórios de comunidade...")
    run_reports()

    print("[KG] 5/5 Embeddings (TextUnit, entidades, relatórios — necessário para recuperação)...")
    run_embed_all()

    print("[KG] Indexação concluída.")
    TRACKER.print_summary()

    # Cache de chunks locais (gabarito) pode ficar desatualizado se estratégia mudou — limpar
    _chunk_texts_and_tu_ids.cache_clear()
else:
    print("[KG] RUN_KG_INDEXING=False — a saltar indexação (use True na primeira execução ou após alterar os .txt).")

ModuleNotFoundError: No module named 'graphrag.monitoring'

## 6. Correr a comparação


In [ ]:
results: list[dict] = []
warn_empty_gold: list[str] = []

for i, row in enumerate(qa_rows):
    gold_ids = gold_text_unit_ids_for_row(row, DOCS_DIR)
    if not gold_ids:
        warn_empty_gold.append(row.qid or f"linha {i + 1}")

    tag = str(abs(hash(row.question)) % 10_000_000)

    n_ids, n_texts = naive_retrieve_top_k(row.question, K)
    if gold_ids:
        n_rec = recall_at_k(n_ids, gold_ids, K)
        n_prec = precision_at_k(n_ids, gold_ids, K)
    else:
        n_rec, n_prec = float("nan"), float("nan")

    n_ans = naive_generate(row.question, n_texts)
    n_f1 = token_f1(row.reference_answer, n_ans)

    g_state = run_graphrag(row.question, tag)
    g_ids = graphrag_top_k_text_unit_ids(g_state, K)
    g_ans = (g_state.get("final_answer") or "").strip()
    if gold_ids:
        g_rec = recall_at_k(g_ids, gold_ids, K)
        g_prec = precision_at_k(g_ids, gold_ids, K)
    else:
        g_rec, g_prec = float("nan"), float("nan")
    g_f1 = token_f1(row.reference_answer, g_ans)

    results.append(
        {
            "qid": row.qid,
            "q": i + 1,
            "hop": row.tipo_pergunta,
            "system": "naive_rag",
            "recall@5": n_rec,
            "precision@5": n_prec,
            "token_f1": n_f1,
            "question": row.question,
            "reference": row.reference_answer,
            "generated": n_ans,
            "retrieved_ids": " | ".join(n_ids),
            "gold_tu_count": len(gold_ids),
        }
    )
    results.append(
        {
            "qid": row.qid,
            "q": i + 1,
            "hop": row.tipo_pergunta,
            "system": "graphrag",
            "recall@5": g_rec,
            "precision@5": g_prec,
            "token_f1": g_f1,
            "question": row.question,
            "reference": row.reference_answer,
            "generated": g_ans,
            "retrieved_ids": " | ".join(g_ids),
            "gold_tu_count": len(gold_ids),
        }
    )
    r5n = n_rec if n_rec == n_rec else -1.0
    r5g = g_rec if g_rec == g_rec else -1.0
    print(f"[{i + 1}/{len(qa_rows)}] {row.tipo_pergunta}  naive R@5={r5n:.3f}  graphrag R@5={r5g:.3f}")

if warn_empty_gold:
    print("Aviso: sem TU correspondente ao trecho (gabarito vazio) para qid:", ", ".join(warn_empty_gold))

print("Concluído.")

## 7. Agregar e visualizar


In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None

if pd is not None:
    df = pd.DataFrame(results)
    display_cols = ["q", "qid", "hop", "system", "recall@5", "precision@5", "token_f1", "gold_tu_count"]
    display(df[display_cols])

    summary = (
        df.groupby("system")[["recall@5", "precision@5", "token_f1"]]
        .mean(numeric_only=True)
        .rename_axis("média macro")
    )
    display(summary)
else:
    from statistics import mean

    for sys in ("naive_rag", "graphrag"):
        sub = [r for r in results if r["system"] == sys]
        def mkey(k: str):
            vals = [x[k] for x in sub if isinstance(x[k], (int, float)) and x[k] == x[k]]
            return mean(vals) if vals else float("nan")
        print(sys, "recall@5", mkey("recall@5"), "precision@5", mkey("precision@5"), "token_f1", mkey("token_f1"))